# **LumeOpt**

In [ ]:
import requests
import pandas as pd
import numpy as np
import os
import json

In [ ]:
# Imports para o PCA
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
import random
import tensorflow as tf

random.seed(532)
np.random.seed(532)
tf.random.set_seed(532)

In [ ]:
!pip install unidecode
from unidecode import unidecode

In [ ]:
# Caminho do github onde encontram-se os dataframes estão armazenados
basePL_path = 'https://github.com/LunusMax/epl-seriea-data/tree/main/Premier_League'
baseA_path = 'https://github.com/LunusMax/epl-seriea-data/tree/main/Serie_A'

games_path = 'https://github.com/LunusMax/epl-seriea-data/tree/main/Games'

# Carregamento dos dados dos jogadores

In [ ]:
# Constantes
num_features = 47

In [ ]:
from collections import defaultdict

In [ ]:
players_A = defaultdict(lambda: ['-'] + (num_features-1)*[0]) # se o jogador não for encontrado retorna um vetor de zeros (mais um categórico nulo na primeira posição)
teams_A = {}

players_PL = defaultdict(lambda: ['-'] + (num_features-1)*[0]) # se o jogador não for encontrado retorna um vetor de zeros (mais um categórico nulo na primeira posição)
teams_PL = {}

**CARREGAR PLAYERS EM DICTIONARY:**

In [ ]:
# Configurações de repositório
user = "LunusMax"
repo = "epl-seriea-data"

def get_csv_file_list_from_github(folder):
    api_url = f"https://api.github.com/repos/{user}/{repo}/contents/{folder}"
    response = requests.get(api_url)
    if response.status_code != 200:
        raise Exception(f"Erro ao acessar a API do GitHub ({folder})")
    files = response.json()
    return [f["name"] for f in files if f["name"].endswith(".csv")]

# Função para processar cada liga
def process_league_from_github(folder, league_label):
    file_list = get_csv_file_list_from_github(folder)
    base_url = f"https://raw.githubusercontent.com/{user}/{repo}/main/{folder}"
    players_dict, teams_dict = {}, {}

    for file in file_list:
        name_parts = file.split('_')
        if len(name_parts) < 3:
            print(f"Unexpected file format, skipping: {file}")
            continue

        team = unidecode(name_parts[1].lower().strip())
        season = name_parts[2].replace('.csv', '')

        file_url = f"{base_url}/{file}"
        try:
            data = pd.read_csv(file_url)
        except Exception as e:
            print(f"Error loading {file}: {e}")
            continue

        if "Min." not in data.columns or "Jogador" not in data.columns:
            print(f"Skipping {file} - missing required columns")
            continue

        feature_names = list(data.columns)
        feature_names.remove("Jogador")

        data = data[data["Min."] > 0].sort_values(by="Min.", ascending=False).head(20)
        teams_dict[(team, season)] = []

        for _, row in data.iterrows():
            player = unidecode(row['Jogador'].lower().strip())
            stats = row.drop('Jogador').tolist()
            teams_dict[(team, season)].append(player)
            players_dict[(player, season, team)] = [team] + stats

    filtered_players = [[p, s] + st for (p, s, t), st in players_dict.items()]
    columns = ["Player", "Season", "Team"] + feature_names
    df = pd.DataFrame(filtered_players, columns=columns)
    print(f"{league_label}: {len(df)} jogadores processados.")
    return df

In [ ]:
# Processar as ligas
df_A = process_league_from_github("Serie_A", "Serie A")
df_PL = process_league_from_github("Premier_League", "Premier League")

In [ ]:
# Juntar DataFrames e armazenar
df_Players = pd.concat([df_A, df_PL], ignore_index=True)

In [ ]:
# =========================
# MARKET VALUE (PUBLIC INTERFACE)
# =========================

try:
    from src.market_value import compute_market_value
    print("✅ Using PRIVATE market value module")

except ImportError:
    print("⚠️ Using PUBLIC simplified market value")

    def compute_market_value(df_players):
        """
        Versão pública simplificada (não reflete a implementação original).
        """

        # Proxy determinístico (não usar random!)
        df_players['Valor_Mercado'] = (
            (30 - df_players['Idade']).clip(lower=0) * 500_000
        )

        return df_players


# Aplicação
df_Players = compute_market_value(df_Players)

In [ ]:
df_Players.head(10)

In [ ]:
# Count the number of players for each team and season combination in df_PL_pca
player_count = df_PL.groupby(['Team', 'Season']).size().reset_index(name='player_count')

# Display the resulting DataFrame
display(player_count)

# Correção dos nomes dos times

In [ ]:
def replace_team_names(df, aliases):
    for alias_list in aliases:
        canonical_name = alias_list[0]
        aliases_to_replace = alias_list[1:]
        df['Team'] = df['Team'].replace(aliases_to_replace, canonical_name)
    return df

In [ ]:
team_aliases_A = [
    ["hellas verona", "hellas-verona"],
    ["inter", "internazionale"]
]

df_A = replace_team_names(df_A, team_aliases_A)
df_Players = replace_team_names(df_Players, team_aliases_A)

In [ ]:
team_aliases_PL = [
    ["crystal-palace", "crystal palace"],
    ["aston-villa", "aston villa"],
    ["leicester-city", "leicester city"],
    ["manchester-city", "manchester city"],
    ["manchester-united", "manchester united", "manchester-utd", "manchester utd"],
    ["brighton-and-hove-albion", "brighton"],
    ["tottenham-hotspur", "tottenham"],
    ["nottingham-forest", "nott'ham forest"],
    ["leeds-united", "leeds united"],
    ["wolverhampton-wanderers", "wolves"],
    ["luton-town", "luton town"],
    ["west-ham-united", "west ham"],
    ["newcastle-united", "newcastle utd"],
    ["sheffield-united", "sheffield utd"],
    ["west-brom", "west brom"],
    ["cardiff", "cardiff city", "cardiff-city"],
    ["norwich", "norwich-city", "norwich city"]
]

df_PL = replace_team_names(df_PL, team_aliases_PL)
df_Players = replace_team_names(df_Players, team_aliases_PL)

In [ ]:
df_PL.head(20)

In [ ]:
# Contar o número de jogadores para cada time e temporada combinados no df_PL_pca
player_count = df_PL.groupby(['Team', 'Season']).size().reset_index(name='player_count')

# Mostrar dataframe final
display(player_count)

# **Aplicar PCA para reduzir dimensionalidade**

In [ ]:
# Remover colunas categóricas
numeric_columns = df_A.select_dtypes(include=['number']).columns.tolist()

In [ ]:
# Normalizar os dados
scaler = StandardScaler()
df_A_scaled = scaler.fit_transform(df_A[numeric_columns])

In [ ]:
# Fazer scree plot

# Aplicar PCA com todas as dimensões para análise
pca_full = PCA()
pca_full.fit(df_A_scaled)

# Variância explicada acumulada
explained_variance = np.cumsum(pca_full.explained_variance_ratio_)

# Plotar gráfico de variância explicada
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(explained_variance) + 1), explained_variance, marker='o', linestyle='--')
plt.xlabel("Número de Componentes Principais")
plt.ylabel("Variância Explicada Acumulada")
plt.title("Escolha do Número Ideal de Componentes")
plt.grid(True)
plt.show()

In [ ]:
parametro_pca = 20

In [ ]:
# Aplicar PCA
pca = PCA(n_components=parametro_pca)
principal_components = pca.fit_transform(df_A_scaled)

In [ ]:
# Criar um DataFrame com os componentes principais
df_A_pca = pd.DataFrame(principal_components, columns=[f'PC{i+1}' for i in range(parametro_pca)])

df_A_pca.insert(0, "Player", df_A["Player"].values)
df_A_pca.insert(1, "Season", df_A["Season"].values)
df_A_pca.insert(2, "Team", df_A["Team"].values)

In [ ]:
# Visualizar a variância explicada
explained_variance = pca.explained_variance_ratio_
total_explained_variance = explained_variance.sum()
print(f"Variância explicada total: {total_explained_variance * 100:.2f}%")

In [ ]:
df_A_pca

# **Carregar dados dos jogos e converter em pontos de cada time por temporada**

In [ ]:
# Nome da pasta no GitHub
games_folder = "Games"
base_url = "https://raw.githubusercontent.com/LunusMax/epl-seriea-data/main/Games"
api_url = f"https://api.github.com/repos/LunusMax/epl-seriea-data/contents/{games_folder}"

# Obter todos os arquivos CSV da Serie A
response = requests.get(api_url)
files_json = response.json()
file_A_names = [f["name"] for f in files_json if f["name"].startswith("serie_a") and f["name"].endswith("_resultados.csv")]

# Dicionário com os jogos por temporada
all_seasons_data_A = {}

# Ler cada arquivo e organizar
for file_name in file_A_names:
    season = file_name.split('_')[2].replace('.csv','')
    file_url = f"{base_url}/{file_name}"

    season_data = pd.read_csv(file_url)
    games_list = season_data.to_dict(orient='records')

    for d in games_list:
        d['Em casa'] = d['Em casa'].lower().strip()
        d['Visitante'] = d['Visitante'].lower().strip()

    all_seasons_data_A[season] = games_list

# Mostrar os arquivos carregados em uma linha
print("Arquivos carregados:", file_A_names)

In [ ]:
print(all_seasons_data_A)

In [ ]:
print(f"Temporadas carregadas: {list(all_seasons_data_A.keys())}")

In [ ]:
# Calcular quantos pontos cada time fez em cada temporada

# Inicializar dicionário para armazenar a pontuação dos times por temporada
team_A_points = {}

# Percorrer cada temporada nos dados carregados
for season, games in all_seasons_data_A.items():
    # Criar um dicionário para armazenar os pontos de cada time nessa temporada
    season_points = {}

    # Percorrer todos os jogos da temporada
    for game in games:
        home_team = game["Em casa"]
        away_team = game["Visitante"]
        match_result = game["Classe"]  # 1.0 = vitória do mandante, 2.0 = vitória do visitante, 0.0 = empate

        # Inicializar os times no dicionário, se ainda não existirem
        if home_team not in season_points:
            season_points[home_team] = 0
        if away_team not in season_points:
            season_points[away_team] = 0

        # Definir a pontuação com base no resultado do jogo
        if match_result == 1.0:
            season_points[home_team] += 3  # Vitória do mandante
        elif match_result == 2.0:
            season_points[away_team] += 3  # Vitória do visitante
        elif match_result == 0.0:
            season_points[home_team] += 1  # Empate
            season_points[away_team] += 1  # Empate

    # Armazenar a pontuação total da temporada
    team_A_points[season] = season_points

# Converter o dicionário para um DataFrame para facilitar a análise
df_A_points = pd.DataFrame([
    {"Season": season, "Team": team, "Points": points}
    for season, teams in team_A_points.items()
    for team, points in teams.items()
])


In [ ]:
df_A_points = replace_team_names(df_A_points, team_aliases_A)

In [ ]:
# Visualizar as primeiras linhas
df_A_points.head(21)

In [ ]:
# Nome da pasta no GitHub
games_folder = "Games"
base_url = "https://raw.githubusercontent.com/LunusMax/epl-seriea-data/main/Games"
api_url = f"https://api.github.com/repos/LunusMax/epl-seriea-data/contents/{games_folder}"

# Obter todos os arquivos CSV da Premier League
response = requests.get(api_url)
files_json = response.json()
file_PL_names = [f["name"] for f in files_json if f["name"].startswith("premier_league") and f["name"].endswith("_resultados.csv")]

# Inicializar dicionário para armazenar as temporadas
all_seasons_data_PL = {}

# Carregar os arquivos e organizar por temporadas
for file_name in file_PL_names:
  # Extrair a temporada do nome do arquivo
  season = file_name.split('_')[2].replace('.csv','')
  file_url = f"{base_url}/{file_name}"
  season_data = pd.read_csv(file_url)

  # Converter DataFrame para lista de dicionários
  games_list = season_data.to_dict(orient='records')

  for d in games_list:
    d['Em casa'] = d['Em casa'].lower().strip()  # remoção de acentos, etc
    d['Visitante'] = d['Visitante'].lower().strip()  # remoção de acentos, etc

  all_seasons_data_PL[season] = games_list

# Mostrar os arquivos carregados em uma linha
print("Arquivos carregados:", file_PL_names)

In [ ]:
# Calcular quantos pontos cada time fez em cada temporada

# Inicializar dicionário para armazenar a pontuação dos times por temporada
team_PL_points = {}

# Percorrer cada temporada nos dados carregados
for season, games in all_seasons_data_PL.items():
    # Criar um dicionário para armazenar os pontos de cada time nessa temporada
    season_points = {}

    # Percorrer todos os jogos da temporada
    for game in games:
        home_team = game["Em casa"]
        away_team = game["Visitante"]
        match_result = game["Classe"]  # 1.0 = vitória do mandante, 2.0 = vitória do visitante, 0.0 = empate

        # Inicializar os times no dicionário, se ainda não existirem
        if home_team not in season_points:
            season_points[home_team] = 0
        if away_team not in season_points:
            season_points[away_team] = 0

        # Definir a pontuação com base no resultado do jogo
        if match_result == 1.0:
            season_points[home_team] += 3  # Vitória do mandante
        elif match_result == 2.0:
            season_points[away_team] += 3  # Vitória do visitante
        elif match_result == 0.0:
            season_points[home_team] += 1  # Empate
            season_points[away_team] += 1  # Empate

    # Armazenar a pontuação total da temporada
    team_PL_points[season] = season_points

df_PL_points = pd.DataFrame([
    {"Season": season, "Team": team, "Points": points}
    for season, teams in team_PL_points.items()
    for team, points in teams.items()
])

In [ ]:
df_PL_points = replace_team_names(df_PL_points, team_aliases_PL)

print("\ndf_PL_points after replacement:")
print(df_PL_points.head())

In [ ]:
# Visualizar as primeiras linhas
df_PL_points.head(21)

In [ ]:
df_PL_scaled = scaler.transform(df_PL[numeric_columns])
df_PL_pca = pca.transform(df_PL_scaled)

In [ ]:
df_PL_pca = pd.DataFrame(df_PL_pca, columns=[f'PC{i+1}' for i in range(parametro_pca)])
df_PL_pca.insert(0, "Player", df_PL["Player"].values)
df_PL_pca.insert(1, "Season", df_PL["Season"].values)
df_PL_pca.insert(2, "Team", df_PL["Team"].values)

In [ ]:
# Criar df_Players_pca concatenando df_PL_pca e df_A_pca

df_Players_pca = pd.concat([df_PL_pca, df_A_pca], ignore_index=True)
df_Players_pca.head()

# Os dados dos jogadores devem ser os dados da temporada anterior

In [ ]:
# Contar o número de jogadores para cada time e temporada combinados no df_PL_pca
player_counts_PL_pca = df_PL_pca.groupby(['Team', 'Season']).size().reset_index(name='player_count')

# Mostrar resultados -> tem que haver 20 jogadores
display(player_counts_PL_pca)

In [ ]:
# Definir uma função para calcular a temporada anterior
def get_previous_season(season):
    year1, year2 = map(int, season.split('-'))
    return f"{year1 - 1}-{year2 - 1}"

# Listar as colunas do PC
pc_columns = [f"PC{i}" for i in range(1, 21)]

# Fazer o 'shift' das pontuações identificando as colunas
df_PL_pca_shifted = df_PL_pca[['Team', 'Season', 'Player']].copy()

# Adicionar uma coluna temporária para a temporada anterior
df_PL_pca_shifted['prev_season'] = df_PL_pca_shifted['Season'].apply(get_previous_season)

# Juntar as colunas da temporada anterior
df_PL_pca_shifted = df_PL_pca_shifted.merge(
    df_PL_pca[['Team', 'Player', 'Season'] + pc_columns],
    left_on=['Team', 'Player', 'prev_season'],
    right_on=['Team', 'Player', 'Season'],
    how='left'
)

In [ ]:
df_PL_pca_shifted

In [ ]:
# Limpar colunas
df_PL_pca_shifted = df_PL_pca_shifted.drop(columns=['prev_season', 'Season_y'])
df_PL_pca_shifted = df_PL_pca_shifted.rename(columns={'Season_x': 'Season'})

# Preencher valores nulos com 0
df_PL_pca_shifted[pc_columns] = df_PL_pca_shifted[pc_columns].fillna(0)

# Identificar a primeira temporada no df_PL_pca
first_season = df_PL_pca['Season'].min()

# Remover todas as entradas do df_PL_pca_shifter onde a temporada é a primeira
df_PL_pca_shifted = df_PL_pca_shifted[df_PL_pca_shifted['Season'] != first_season]

In [ ]:
df_PL_pca_shifted

# Data augmentation

In [ ]:
# Criar uma cópia das colunas iniciais com valor 0
df_PL_pca_shifted['copy'] = 0

# Listar dfs (original + cópia)
dfs = [df_PL_pca_shifted]

# Criar 10 cópias com colunas sortidas
for i in range(1, 11):
    # Criar cópia do df original
    df_copy = df_PL_pca_shifted.copy()

    df_copy['copy'] = i
    df_copy = df_copy.sample(frac=1, random_state=i).reset_index(drop=True)

    # Adicionar à lista dos dfs
    dfs.append(df_copy)

# Concatenar dataframes
df_PL_augmented = pd.concat(dfs, ignore_index=True)

In [ ]:
df_PL_augmented

# Concatena os atributos dos jogadores

In [ ]:
try:
    from src.team_season_representation import build_team_season_dataframe
    print("✅ Using PRIVATE team-season representation module")

except ImportError:
    print("⚠️ Using PUBLIC simplified team-season representation module")

    def build_team_season_dataframe(df_augmented, n_components=20):
        """
        Simplified public version.
        Does not reproduce the private augmentation/pivot strategy.
        """

        pc_columns = [col for col in df_augmented.columns if col.startswith("PC")]

        df_team_season = (
            df_augmented[["Season", "Team"] + pc_columns]
            .groupby(["Season", "Team"], as_index=False)
            .mean()
            .fillna(0)
        )

        return df_team_season


df_PL_season = build_team_season_dataframe(df_PL_augmented, n_components=20)

In [ ]:
df_PL_season

In [ ]:
'''df_PL_model = df_PL_season.merge(
    df_PL_points_shifted[['Team', 'Season', 'Points']],
    on=['Team', 'Season'],
    how='left'
)
'''
df_PL_model = df_PL_season.merge(df_PL_points, on=["Season", "Team"])

df_PL_model

In [ ]:
df_PL_model = df_PL_model.drop(columns=['copy'], errors='ignore')

# Adiciona pontos da temporada anterior

In [ ]:
# Adicionar coluna temporária para temporada anterior
df_PL_model['prev_season'] = df_PL_model['Season'].apply(get_previous_season)

# Buscar pontos da temporada anterior
df_PL_model = df_PL_model.merge(
    df_PL_points[['Team', 'Season', 'Points']].rename(columns={'Points': 'Points_prev'}),
    left_on=['Team', 'prev_season'],
    right_on=['Team', 'Season'],
    how='left'
)

# Limpar colunas e jogar fora desnecessárias
df_PL_model = df_PL_model.drop(columns=['prev_season', 'Season_y'])
df_PL_model = df_PL_model.rename(columns={'Season_x': 'Season'})

df_PL_model = df_PL_model.fillna(0)

In [ ]:
df_PL_model

# Repete tudo para serie A

In [ ]:
pc_columns = [f"PC{i}" for i in range(1, 21)]

df_A_pca_shifted = df_A_pca[['Team', 'Season', 'Player']].copy()

df_A_pca_shifted['prev_season'] = df_A_pca_shifted['Season'].apply(get_previous_season)

df_A_pca_shifted = df_A_pca_shifted.merge(
    df_A_pca[['Team', 'Player', 'Season'] + pc_columns],
    left_on=['Team', 'Player', 'prev_season'],
    right_on=['Team', 'Player', 'Season'],
    how='left'
)


df_A_pca_shifted = df_A_pca_shifted.drop(columns=['prev_season', 'Season_y'])
df_A_pca_shifted = df_A_pca_shifted.rename(columns={'Season_x': 'Season'})

df_A_pca_shifted[pc_columns] = df_A_pca_shifted[pc_columns].fillna(0)

first_season = df_A_pca['Season'].min()

df_A_pca_shifted = df_A_pca_shifted[df_A_pca_shifted['Season'] != first_season]

In [ ]:
df_A_pca_shifted['copy'] = 0

dfs = [df_A_pca_shifted]

K = 200
for i in range(1, K+1):
    df_copy = df_A_pca_shifted.copy()
    df_copy['copy'] = i
    df_copy = df_copy.sample(frac=1, random_state=i).reset_index(drop=True)
    dfs.append(df_copy)

df_A_augmented = pd.concat(dfs, ignore_index=True)

In [ ]:
try:
    from src.team_season_representation import build_team_season_dataframe
    print("✅ Using PRIVATE team-season representation module")

except ImportError:
    print("⚠️ Using PUBLIC simplified team-season representation module")

    def build_team_season_dataframe(df_augmented, n_components=20):
        """
        Public simplified version (no augmentation logic).
        """

        pc_columns = [col for col in df_augmented.columns if col.startswith("PC")]

        df_team_season = (
            df_augmented[["Season", "Team"] + pc_columns]
            .groupby(["Season", "Team"], as_index=False)
            .mean()
            .fillna(0)
        )

        return df_team_season


df_A_season = build_team_season_dataframe(df_A_augmented, n_components=20)

In [ ]:
df_A_model = df_A_season.merge(df_A_points, on=["Season", "Team"])
df_A_model = df_A_model.drop(columns=['copy'], errors='ignore')
df_A_model

In [ ]:
df_A_model['prev_season'] = df_A_model['Season'].apply(get_previous_season)

df_A_model = df_A_model.merge(
    df_A_points[['Team', 'Season', 'Points']].rename(columns={'Points': 'Points_prev'}),
    left_on=['Team', 'prev_season'],
    right_on=['Team', 'Season'],
    how='left'
)

df_A_model = df_A_model.drop(columns=['prev_season', 'Season_y'])
df_A_model = df_A_model.rename(columns={'Season_x': 'Season'})

df_A_model = df_A_model.fillna(0)
df_A_model

In [ ]:
# Printar valores nulos

# Contar valores nulos para cada coluna
nan_counts_per_row = df_A_model.isnull().sum(axis=1)

print("Number of NaN values in each row:")
nan_counts_per_row.sum()

# Treinamento

In [ ]:
df_A_model = df_A_model[sorted(df_A_model.columns)]
df_PL_model = df_PL_model[sorted(df_PL_model.columns)]

In [ ]:
df_A_model

In [ ]:
# Separar features (X) e target (y) para o modelo PL
X_A_train = df_A_model.drop(columns=["Season", "Team", "Points"])  # Features (PC1 a PC10)
y_A_train = df_A_model["Points"]  # Variável alvo (pontuação final)

# Separar features (X) e target (y) para os dados de teste da Serie A
X_PL_test = df_PL_model.drop(columns=["Season", "Team", "Points"])  # Features (PC1 a PC10)
y_PL_test = df_PL_model["Points"]  # Variável alvo (pontuação final)

In [ ]:
scaler = StandardScaler()
y_A_train_norm = scaler.fit_transform(y_A_train.values.reshape(-1, 1)).flatten() #PODE SER PROBLEMÁTICO
y_PL_test_norm = scaler.transform(y_PL_test.values.reshape(-1, 1)).flatten()

In [ ]:
np.mean(y_A_train_norm), np.std(y_A_train_norm)

In [ ]:
try:
    from src.model import create_model
    print("✅ Using PRIVATE model architecture")

except ImportError:
    print("⚠️ Using PUBLIC simplified model")

    from tensorflow import keras
    from tensorflow.keras import layers

    def create_model(input_dim):
        """
        Simplified public model (does not reflect final architecture)
        """

        model = keras.Sequential([
            layers.Input(shape=(input_dim,)),
            layers.Dense(64, activation='relu'),
            layers.Dense(1)
        ])

        return model

In [ ]:
X_A_train.shape

In [ ]:
X_PL_test.shape

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, Callback
import tensorflow as tf
#from sklearn.metrics import r2_score


model = create_model(X_A_train.shape[1])
model.compile(optimizer='adam', loss='mse', metrics = ['mae', 'mse', tf.keras.metrics.R2Score()])
#model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics = ['mae', 'mse', tf.keras.metrics.R2Score()])


# Callback do Keras que restaura os melhores pesos
early_stop = EarlyStopping(
    monitor='val_r2_score',
    patience=10,
    restore_best_weights=True
)

# Treinamento
history = model.fit(
    X_A_train,
    y_A_train_norm,
    batch_size=32,
    epochs=1000,
    validation_data=(X_PL_test, y_PL_test_norm),
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
results = model.evaluate(X_PL_test, y_PL_test_norm, verbose=0)
print("Best R2:", results[3])

In [ ]:
# =========================
# TEAM REPRESENTATION (PUBLIC INTERFACE)
# =========================

try:
    from src.team_representation import create_team_pca_dataframe
    print("✅ Using PRIVATE team representation module")

except ImportError:
    print("⚠️ Using PUBLIC simplified team representation")

    def create_team_pca_dataframe(team, season, L, player_attributes, player_attributes_PCA, points):
        """
        Simplified public version of team representation.
        Does NOT reflect the full private implementation.
        """

        # Simplified roster selection (no business logic)
        team_players = player_attributes[
            player_attributes['Team'] == team
        ]['Player'].head(20).tolist()

        # Guarantee 20 players
        final_roster = (team_players + L)[:20]

        # Create PCA structure (keeps shape, hides logic)
        pca_columns = [f'PC{i}_{j}' for i in range(1, 21) for j in range(1, 21)]
        result_df = pd.DataFrame(columns=pca_columns, dtype=np.float64)

        for j, player in enumerate(final_roster, 1):

            player_pca = player_attributes_PCA[
                player_attributes_PCA['Player'] == player
            ][[f'PC{i}' for i in range(1, 21)]]

            if player_pca.empty:
                player_data = np.zeros(20)
            else:
                player_data = player_pca.iloc[0].values

            for i, value in enumerate(player_data, 1):
                result_df.loc[0, f'PC{i}_{j}'] = value

        # Simplified points
        result_df['Points_prev'] = 0.0

        return result_df

In [ ]:
try:
    from src.predict import preve_pontos
    print("✅ Using PRIVATE prediction model")

except ImportError:
    print("⚠️ Using PUBLIC simplified prediction")

    def preve_pontos(team, season, new_players, player_attributes, player_attributes_PCA, points,
                     model=None,
                     scaler=None,):
        """
        Simplified public version of prediction function.
        """

        team_df = create_team_pca_dataframe(
            team, season, new_players,
            player_attributes, player_attributes_PCA, points
        )

        # Proxy simples: soma dos PCs
        return float(team_df.sum(axis=1).values[0] * 0.01)


In [ ]:
df_all_players_pca = pd.concat([df_PL_pca, df_A_pca], ignore_index=True)
df_all_players = df_Players

In [ ]:
preve_pontos(
    team='cagliari',
    season='2021-2022',
    new_players=[],
    player_attributes=df_all_players,
    player_attributes_PCA=df_all_players_pca,
    points=df_A_points,
    model=model,
    scaler=scaler,
)

GERAR A TABELA DE ATRIBUTOS FILTRADOS AQUI

In [ ]:
df_player_prev = df_Players[df_Players['Season']=='2020-2021']
df_player_prev_pca = df_Players_pca[df_Players_pca['Season']=='2020-2021']

Guloso First

In [ ]:
try:
    from src.optimization import guloso
    print("✅ Using PRIVATE optimization module")

except ImportError:
    print("⚠️ Using PUBLIC simplified optimization module")

    def guloso(
        team,
        season,
        player_attributes,
        player_attributes_PCA,
        points,
        budget,
        predict_function,
        verbose=True,
    ):
        evaluations = 0
        iterations = 0

        price_map = player_attributes.set_index("Player")["Valor_Mercado"].to_dict()
        player_names = player_attributes["Player"].unique().tolist()

        current_new_players = []
        current_score = predict_function(
            team,
            season,
            current_new_players,
            player_attributes,
            player_attributes_PCA,
            points,
        )
        current_cost = 0

        for p in player_names[:10]:
            player_price = price_map.get(p)
            if player_price is None:
                continue

            if current_cost + player_price <= budget:
                current_new_players.append(p)
                current_cost += player_price
                evaluations += 1

        iterations = 1
        return current_new_players, current_cost, current_score, iterations, evaluations

In [ ]:
def preve_pontos_modelo(
    team,
    season,
    new_players,
    player_attributes,
    player_attributes_PCA,
    points,
):
    return preve_pontos(
        team=team,
        season=season,
        new_players=new_players,
        player_attributes=player_attributes,
        player_attributes_PCA=player_attributes_PCA,
        points=points,
        model=model,
        scaler=scaler,
    )

In [ ]:
guloso(team='cagliari',
       season='2021-2022',
       player_attributes=df_player_prev,
       player_attributes_PCA=df_player_prev_pca,
       points=df_A_points,
       budget=60000000,
       predict_function=preve_pontos_modelo)

In [ ]:
try:
    from src.optimization import guloso_best
    print("✅ Using PRIVATE greedy best module")

except ImportError:
    print("⚠️ Using PUBLIC simplified greedy best module")

    def guloso_best(
        team,
        season,
        player_attributes,
        player_attributes_PCA,
        points,
        budget,
        predict_function,
        verbose=True,
        progress=True,
    ):
        evaluations = 0
        iterations = 1

        price_map = player_attributes.set_index("Player")["Valor_Mercado"].to_dict()
        player_names = player_attributes["Player"].unique().tolist()

        current_new_players = []
        current_score = predict_function(
            team,
            season,
            current_new_players,
            player_attributes,
            player_attributes_PCA,
            points,
        )
        current_cost = 0

        for p in player_names[:10]:
            player_price = price_map.get(p)

            if player_price is None:
                continue

            if p not in current_new_players and current_cost + player_price <= budget:
                current_new_players.append(p)
                current_cost += player_price
                evaluations += 1

        return current_new_players, current_cost, current_score, iterations, evaluations

In [ ]:
try:
    from src.optimization import busca_local
    print("✅ Using PRIVATE local search module")

except ImportError:
    print("⚠️ Using PUBLIC simplified local search module")

    def busca_local(
        team,
        season,
        player_attributes,
        player_attributes_PCA,
        points,
        budget,
        predict_function,
        verbose=True,
    ):
        evaluations = 0
        iterations = 1
        improvements = 0

        current_new_players, current_cost, current_score, _, _ = guloso(
            team=team,
            season=season,
            player_attributes=player_attributes,
            player_attributes_PCA=player_attributes_PCA,
            points=points,
            budget=budget,
            predict_function=predict_function,
            verbose=verbose,
        )

        return current_new_players, current_cost, current_score, iterations, evaluations, improvements

In [ ]:
try:
    from src.optimization import busca_local_best
    print("✅ Using PRIVATE local search best module")

except ImportError:
    print("⚠️ Using PUBLIC simplified local search best module")

    def busca_local_best(
        team,
        season,
        player_attributes,
        player_attributes_PCA,
        points,
        budget,
        predict_function,
        verbose=True,
        log_interval=5000,
    ):
        evaluations = 0
        iterations = 1
        improvements = 0

        current_new_players, current_cost, current_score, _, _ = guloso(
            team=team,
            season=season,
            player_attributes=player_attributes,
            player_attributes_PCA=player_attributes_PCA,
            points=points,
            budget=budget,
            predict_function=predict_function,
            verbose=verbose,
        )

        return current_new_players, current_cost, current_score, iterations, evaluations, improvements

# Benchmarking

In [ ]:
import time

def medir_tempo(func, **kwargs):
    start = time.perf_counter()
    result = func(**kwargs)
    end = time.perf_counter()
    runtime = end - start
    return result, runtime

In [ ]:
def extrair_metricas(res):
    return {
        "Score": res[2] if len(res) > 2 else None,
        "Iteracoes": res[3] if len(res) > 3 else None,
        "Avaliacoes": res[4] if len(res) > 4 else None,
        "Melhoras": res[5] if len(res) > 5 else None,
    }

In [ ]:
SEED = 532
random.seed(SEED)

algoritmos = {
    "Guloso First": lambda: guloso(
        team='cagliari',
        season='2021-2022',
        player_attributes=df_player_prev,
        player_attributes_PCA=df_player_prev_pca,
        points=df_A_points,
        budget=60000000,
        predict_function=preve_pontos_modelo
    ),

    "Guloso Best": lambda: guloso_best(
        team='cagliari',
        season='2021-2022',
        player_attributes=df_player_prev,
        player_attributes_PCA=df_player_prev_pca,
        points=df_A_points,
        budget=60000000,
        predict_function=preve_pontos_modelo
    ),

    "Local First": lambda: busca_local(
        team='cagliari',
        season='2021-2022',
        player_attributes=df_player_prev,
        player_attributes_PCA=df_player_prev_pca,
        points=df_A_points,
        budget=60000000,
        predict_function=preve_pontos_modelo
    ),

    "Local Best": lambda: busca_local_best(
        team='cagliari',
        season='2021-2022',
        player_attributes=df_player_prev,
        player_attributes_PCA=df_player_prev_pca,
        points=df_A_points,
        budget=60000000,
        predict_function=preve_pontos_modelo
    ),
}

In [ ]:
registros = []

for nome, chamada in algoritmos.items():
    res, tempo = medir_tempo(chamada)
    m = extrair_metricas(res)

    registros.append({
        "Algoritmo": nome,
        "Tempo (s)": tempo,
        "Iterações": m["Iteracoes"],
        "Avaliações": m["Avaliacoes"],
        "Melhoras": m["Melhoras"],
        "Score final": m["Score"]
    })

df_benchmark_1x = pd.DataFrame(registros)
print(df_benchmark_1x)

In [ ]:
SEED = 532
random.seed(SEED)

algoritmos = {
    "Guloso First": lambda: guloso(
        team='napoli',
        season='2021-2022',
        player_attributes=df_player_prev,
        player_attributes_PCA=df_player_prev_pca,
        points=df_A_points,
        budget=120000000,
        predict_function=preve_pontos_modelo
    ),

    "Guloso Best": lambda: guloso_best(
        team='napoli',
        season='2021-2022',
        player_attributes=df_player_prev,
        player_attributes_PCA=df_player_prev_pca,
        points=df_A_points,
        budget=120000000,
        predict_function=preve_pontos_modelo
    ),

    "Local First": lambda: busca_local(
        team='napoli',
        season='2021-2022',
        player_attributes=df_player_prev,
        player_attributes_PCA=df_player_prev_pca,
        points=df_A_points,
        budget=120000000,
        predict_function=preve_pontos_modelo
    ),

    "Local Best": lambda: busca_local_best(
        team='napoli',
        season='2021-2022',
        player_attributes=df_player_prev,
        player_attributes_PCA=df_player_prev_pca,
        points=df_A_points,
        budget=120000000,
        predict_function=preve_pontos_modelo
    ),
}

In [ ]:
SEED_BASE = 532
n = 3

registros_n = []

for rodada in range(1, n + 1):
    random.seed(SEED_BASE + rodada)

    for nome, chamada in algoritmos.items():
        res, tempo = medir_tempo(chamada)
        m = extrair_metricas(res)

        registros_n.append({
            "Rodada": rodada,
            "Algoritmo": nome,
            "Tempo": tempo,
            "Iteracoes": m["Iteracoes"],
            "Avaliacoes": m["Avaliacoes"],
            "Melhoras": m["Melhoras"],
            "Score": m["Score"]
        })

df_benchmark_n = pd.DataFrame(registros_n)
print(df_benchmark_n)

In [ ]:
preve_pontos(team='cagliari',
             season='2021-2022',
             new_players=[],
             player_attributes=df_all_players,
             player_attributes_PCA=df_all_players_pca,
             points=df_A_points,
            )

In [ ]:
preve_pontos(team='napoli',
             season='2021-2022',
             new_players=[],
             player_attributes=df_all_players,
             player_attributes_PCA=df_all_players_pca,
             points=df_A_points,
            )

In [ ]:
for rodada in range(1, n + 1):
    for nome, chamada in algoritmos.items():
        random.seed(SEED_BASE + rodada)

        res, tempo = medir_tempo(chamada)
        m = extrair_metricas(res)

        registros_n.append({
            "Rodada": rodada,
            "Algoritmo": nome,
            "Tempo": tempo,
            "Iteracoes": m["Iteracoes"],
            "Avaliacoes": m["Avaliacoes"],
            "Melhoras": m["Melhoras"],
            "Score": m["Score"]
        })

df_benchmark_n = pd.DataFrame(registros_n)
print(df_benchmark_n)